### 0. Set vars and MAP drives

In [2]:
import os

%env LOCAL_PROJECT_DIR=local_project_dir

os.environ["LOCAL_DATA_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "data")
os.environ["LOCAL_EXPERIMENT_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "classification")
os.environ["LOCAL_SPECS_DIR"] = os.path.join(os.getcwd(), "specs")

%env KEY=dHFmaTA4amVydDR0cG8xOTcwM2FoZW5vMmI6ZjZhOTRmYzUtMjkzOS00NTk2LThkNWEtYzk0NDM1Y2EzZDg3
%env NUM_GPUS=1
%env SPECS_DIR=/workspace/tao-experiments/classification/specs
%env USER_EXPERIMENT_DIR=/workspace/tao-experiments/classification

env: LOCAL_PROJECT_DIR=local_project_dir
env: KEY=dHFmaTA4amVydDR0cG8xOTcwM2FoZW5vMmI6ZjZhOTRmYzUtMjkzOS00NTk2LThkNWEtYzk0NDM1Y2EzZDg3
env: NUM_GPUS=1
env: SPECS_DIR=/workspace/tao-experiments/classification/specs
env: USER_EXPERIMENT_DIR=/workspace/tao-experiments/classification


In [22]:
# Mapping up the local directories to the TAO docker.
import json
import os
mounts_file = os.path.expanduser("~/.tao_mounts.json")

# Define the dictionary with the mapped drives
drive_map = {
    "Mounts": [
        # Mapping the data directory
        {
            "source": os.environ["LOCAL_PROJECT_DIR"],
            "destination": "/workspace/tao-experiments"
        },
        # Mapping the specs directory.
        {
            "source": os.environ["LOCAL_SPECS_DIR"],
            "destination": os.environ["SPECS_DIR"]
        },
    ],
    "DockerOptions":{
        "user": "{}:{}".format(os.getuid(), os.getgid())
    }
}

# Writing the mounts file.
with open(mounts_file, "w") as mfile:
    json.dump(drive_map, mfile, indent=4)

In [23]:
!cat ~/.tao_mounts.json

{
    "Mounts": [
        {
            "source": "local_project_dir",
            "destination": "/workspace/tao-experiments"
        },
        {
            "source": "/home/kmilo8346/pactolus_belt_vision/classification/specs",
            "destination": "/workspace/tao-experiments/classification/specs"
        }
    ],
    "DockerOptions": {
        "user": "1001:1002"
    }
}

In [19]:
# Showing list of specification files.
!ls -rlt $LOCAL_SPECS_DIR

total 8
-rw-rw-r-- 1 kmilo8346 kmilo8346 1272 Feb 24 15:13 stage1_classification_spec.cfg
-rw-rw-r-- 1 kmilo8346 kmilo8346 1142 Feb 24 15:14 stage1_classification_retrain_spec.cfg


### 1. Download && Prepare Darkstore Dataset

In [5]:
# got culr command with dev tools in browser from "https://docs.google.com/uc?export=download&id=1-lzgo4lD90e1UQdKzRjRoyQXtI10K9YD&confirm=t"
!mkdir -p $LOCAL_DATA_DIR
!curl 'https://doc-0k-34-docs.googleusercontent.com/docs/securesc/2jblqmi0vne0t7e51j6alkc1o9fnk7an/me640jb60no8rc5nqedrjtfdefmnrjv1/1645648350000/05180970142852220550/15595806178400486391/1-lzgo4lD90e1UQdKzRjRoyQXtI10K9YD?e=download&ax=ACxEAsZmpojTlv3eX71nBO4De376LWcyfFV_XAklLgb_VTywjWHqEtW5Pl_Pw0EfcbY9vThRaQqLKCw_Wo9E4a9y4d_ReHj3yPOWXLC7hxEMw7EtrGBLqjan3HVt6_4ug6GHLUmM8qCKpvVbTM6IbkgFPYpqQU6MTA5x4CSbE10HMoP7HZsrvH4sLx2jtaHcT1iuP6YN3rEMReSvnzpr5YJx8hJn8ID7_5XZi8u0nDaav5DVH5Qqjji75BXN8HvXDxRX96LktY-0CIz4nsOXYxbMu6qI8xA-UVOfQPmy5taGSE5nc4OYlVGiAQHnfQeYFGUvjZnVn1WYhuW4n4ikfM3s0K7DLChbn2ZlP2qXE4zP2l9iRoUZWe0VkPMTAxgGmW7f0-HuWgKDZUjBAZ6f5JHetrm9tVwzswMCDe5raUI9sqf1qef7kS9MlC2te48FNv7f9c27WTKC7XN4z_YXqwU0r9ZckWZvc3whZhEHilf23mG7GyVb6yIqKI4AX7J9pXXgMARRbNdiLv3sRrFW-mD6klM6miGDypmyK9F1wDOvFa58E8HOruUhW9tdrREJPZEK0Ve-Fzz2VRDv39-IxOh9BmKc8gFspDhJXo1eFAv7vohE7yhimiHScp875DeH6lGIGvt-STm-yhzO4H7UZRRrmx0d7FnH5GUwkHJiRIA&authuser=0' -H 'authority: doc-0k-34-docs.googleusercontent.com' -H 'upgrade-insecure-requests: 1' -H 'user-agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/98.0.4758.102 Safari/537.36' -H 'accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9' -H 'sec-gpc: 1' -H 'sec-fetch-site: none' -H 'sec-fetch-mode: navigate' -H 'sec-fetch-user: ?1' -H 'sec-fetch-dest: document' -H 'accept-language: en-US,en;q=0.9' -H 'cookie: AUTH_3ceuna1oj9pjur5cq30hsptg7d9vispv=15595806178400486391|1645648350000|l6qr1l3l5blqq1e1hr7a5dd0j6e7p7mm' --compressed --output $LOCAL_DATA_DIR/darkstore_dataset_v1.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 22.2G  100 22.2G    0     0   152M      0  0:02:29  0:02:29 --:--:--  166M49M      0  0:02:32  0:00:50  0:01:42  174M


In [14]:
# Check the dataset is present
!mkdir -p $LOCAL_DATA_DIR
!if [ ! -f $LOCAL_DATA_DIR/darkstore_dataset_v1.zip ]; then echo 'dataset zip file not found, please download.'; else echo 'Found dataset zip file.';fi

Found dataset zip file.


In [15]:
# unpack
!unzip -u $LOCAL_DATA_DIR/darkstore_dataset_v1.zip -d $LOCAL_DATA_DIR/training

unzip:  cannot find or open local_project_dir/data/darkstore_dataset_v1.zip, local_project_dir/data/darkstore_dataset_v1.zip.zip or local_project_dir/data/darkstore_dataset_v1.zip.ZIP.


In [3]:
!ls -l $LOCAL_DATA_DIR/training | wc -l

3251


In [6]:
# get spot scanned
!wget "https://docs.google.com/uc?export=download&id=187vLZAVaDnzTBCOjaRCHYB9zDQWvHC1S&confirm=t" -O $LOCAL_DATA_DIR/spot_box_scanned.zip

--2022-02-23 21:31:00--  https://docs.google.com/uc?export=download&id=187vLZAVaDnzTBCOjaRCHYB9zDQWvHC1S&confirm=t
Resolving docs.google.com (docs.google.com)... 173.194.198.101, 173.194.198.113, 173.194.198.102, ...
Connecting to docs.google.com (docs.google.com)|173.194.198.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://doc-0g-7s-docs.googleusercontent.com/docs/securesc/ha0ro937gcuc7l7deffksulhg5h7mbp1/oouaci80ed2hto1q6eed9rd5o0c8h7pe/1645651800000/05180970142852220550/*/187vLZAVaDnzTBCOjaRCHYB9zDQWvHC1S?e=download [following]
--2022-02-23 21:31:00--  https://doc-0g-7s-docs.googleusercontent.com/docs/securesc/ha0ro937gcuc7l7deffksulhg5h7mbp1/oouaci80ed2hto1q6eed9rd5o0c8h7pe/1645651800000/05180970142852220550/*/187vLZAVaDnzTBCOjaRCHYB9zDQWvHC1S?e=download
Resolving doc-0g-7s-docs.googleusercontent.com (doc-0g-7s-docs.googleusercontent.com)... 74.125.69.132, 2607:f8b0:4001:c08::84
Connecting to doc-0g-7s-docs.googleusercontent.com (doc-0g

In [7]:
# unzip spot_box_scanned
!unzip -u $LOCAL_DATA_DIR/spot_box_scanned.zip -d $LOCAL_DATA_DIR/spot

Archive:  local_project_dir/data/spot_box_scanned.zip
   creating: local_project_dir/data/spot/scanned/
  inflating: local_project_dir/data/spot/scanned/37a6b675-33cb-4ed7-b172-177666585a51|04017100737909|99  
  inflating: local_project_dir/data/spot/scanned/37ee1368-76ad-4c1e-b16f-2421ba8e9c9c|07802410001430|99  
  inflating: local_project_dir/data/spot/scanned/1e07a3c7-e8b2-4b64-ba17-6adadf9e36f2|07805000301293|99  
  inflating: local_project_dir/data/spot/scanned/9080ffb6-b0b5-4520-8dff-fe045d8eb7a1|07613287354013|99  
  inflating: local_project_dir/data/spot/scanned/bc8e64e3-113e-42af-98a5-f22e65891843|07807910012085|99  
  inflating: local_project_dir/data/spot/scanned/ed7122cb-e709-4291-9e7f-a12abc4368c9|07803200804156|99  
  inflating: local_project_dir/data/spot/scanned/2a567bcd-a58d-49f8-b2c6-39ddad11652d|07804627650319|99  
  inflating: local_project_dir/data/spot/scanned/e00552da-8a27-4046-b86a-27153257f29d|07801810804443|10  
  inflating: local_project_dir/data/spot/scanned

In [18]:
!ls -l $LOCAL_DATA_DIR/spot/scanned | wc -l

1113


In [31]:
# script pa ordenar las clases por mas populadas
# del training dataset
import os
from glob import glob

all_eans_dirs = glob(os.path.join(os.getenv('LOCAL_DATA_DIR'), 'training/*'))
all_eans_dirs.sort(key=lambda dir: len(glob(dir + '/*')), reverse=True)
for dir in all_eans_dirs[:50]:
  print(len(glob(dir + '/*')), dir)

9685 local_project_dir/data/training/7802264020304
6295 local_project_dir/data/training/7803520001624
2785 local_project_dir/data/training/7802230082534
139 local_project_dir/data/training/3472710019073
137 local_project_dir/data/training/7802410001423
102 local_project_dir/data/training/7807910030751
94 local_project_dir/data/training/8003180906249
91 local_project_dir/data/training/7802800556427
89 local_project_dir/data/training/013409341155
86 local_project_dir/data/training/013409351000
83 local_project_dir/data/training/7802200039643
81 local_project_dir/data/training/7802247326201
80 local_project_dir/data/training/7804650970422
79 local_project_dir/data/training/7807910027980
79 local_project_dir/data/training/7802245000998
76 local_project_dir/data/training/7802247061355
75 local_project_dir/data/training/7802230082510
75 local_project_dir/data/training/041390050046
73 local_project_dir/data/training/041570068373
67 local_project_dir/data/training/7500435129534
66 local_projec

In [35]:
# prepare to stage 1 training
import shutil
from glob import glob

stage1_dir = os.path.join(os.getenv('LOCAL_DATA_DIR'), 'stage1_training')

for dir in all_eans_dirs[:50]:
  print(f'coping ean {dir.split("/")[-1]}')
  shutil.copytree(dir, os.path.join(stage1_dir, dir.split('/')[-1]))
print("finished")

coping ean 7802264020304
coping ean 7803520001624
coping ean 7802230082534
coping ean 3472710019073
coping ean 7802410001423
coping ean 7807910030751
coping ean 8003180906249
coping ean 7802800556427
coping ean 013409341155
coping ean 013409351000
coping ean 7802200039643
coping ean 7802247326201
coping ean 7804650970422
coping ean 7807910027980
coping ean 7802245000998
coping ean 7802247061355
coping ean 7802230082510
coping ean 041390050046
coping ean 041570068373
coping ean 7500435129534
coping ean 7802225689472
coping ean 7802230082527
coping ean 7805040004116
coping ean 7804658020365
coping ean 7791130001830
coping ean 7896423419344
coping ean 5412956121114
coping ean 4000415358303
coping ean 4001499915222
coping ean 7802225689076
coping ean 8410509000074
coping ean 4011676005646
coping ean 7802800556434
coping ean 8428532230047
coping ean 7891035040191
coping ean 7804660170522
coping ean 4018077686658
coping ean 7501058753458
coping ean 8023678424154
coping ean 7790520024930
copi

In [6]:
# checking
from glob import glob
print(len(glob(os.getenv('LOCAL_DATA_DIR') + '/stage1_training/*')))

50


In [7]:
# install pip requirements
!pip3 install tqdm
!pip3 install matplotlib

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     |████████████████████████████████| 76 kB 5.7 MB/s             
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [11]:
# split training data

import os
import shutil
from tqdm import tqdm
from glob import glob
from random import shuffle

DATA_DIR = os.getenv('LOCAL_DATA_DIR')
SOURCE_DIR = os.path.join(DATA_DIR, 'stage1_training')
TARGET_DIR = os.path.join(DATA_DIR, 'stage1_training_split')

dir_list = next(os.walk(SOURCE_DIR))[1]
for dir_i in tqdm(dir_list):
  newdir_train = os.path.join(TARGET_DIR, 'train', dir_i)
  newdir_val = os.path.join(TARGET_DIR, 'val', dir_i)
  newdir_test = os.path.join(TARGET_DIR, 'test', dir_i)
        
  if not os.path.exists(newdir_train):
    os.makedirs(newdir_train)
  if not os.path.exists(newdir_val):
    os.makedirs(newdir_val)
  if not os.path.exists(newdir_test):
    os.makedirs(newdir_test)

  img_list = glob(os.path.join(SOURCE_DIR, dir_i, '*.jpeg'))
  # shuffle data
  shuffle(img_list)

  for j in range(int(len(img_list)*0.7)):
    shutil.copy2(img_list[j], os.path.join(TARGET_DIR, 'train', dir_i))

  for j in range(int(len(img_list)*0.7), int(len(img_list)*0.8)):
    shutil.copy2(img_list[j], os.path.join(TARGET_DIR, 'val', dir_i))
                
  for j in range(int(len(img_list)*0.8), len(img_list)):
    shutil.copy2(img_list[j], os.path.join(TARGET_DIR, 'test', dir_i))
                
print('Done splitting train dataset.')

100%|██████████| 50/50 [01:42<00:00,  2.06s/it]

Done splitting train dataset.


In [18]:
# count
!find $LOCAL_DATA_DIR/stage1_training_split/train -type f | wc -l
!find $LOCAL_DATA_DIR/stage1_training_split/val -type f | wc -l
!find $LOCAL_DATA_DIR/stage1_training_split/test -type f | wc -l

15406
2206
4424


### 2. Installing the TAO launcher

In [ ]:
# SKIP this cell IF you have already installed the TAO launcher.
!pip3 install nvidia-pyindex
!pip3 install nvidia-tao

In [24]:
# View the versions of the TAO launcher
!tao info

Configuration of the TAO Toolkit Instance
dockers: ['nvidia/tao/tao-toolkit-tf', 'nvidia/tao/tao-toolkit-pyt', 'nvidia/tao/tao-toolkit-lm']
format_version: 2.0
toolkit_version: 3.21.11
published_date: 11/08/2021


### 3. Download pre-trained models && TRAIN

In [25]:
# Installing NGC CLI on the local machine.
## Download and install
%env CLI=ngccli_cat_linux.zip
!mkdir -p $LOCAL_PROJECT_DIR/ngccli

# Remove any previously existing CLI installations
!rm -rf $LOCAL_PROJECT_DIR/ngccli/*
!wget "https://ngc.nvidia.com/downloads/$CLI" -P $LOCAL_PROJECT_DIR/ngccli
!unzip -u "$LOCAL_PROJECT_DIR/ngccli/$CLI" -d $LOCAL_PROJECT_DIR/ngccli/
!rm $LOCAL_PROJECT_DIR/ngccli/*.zip 
os.environ["PATH"]="{}/ngccli:{}".format(os.getenv("LOCAL_PROJECT_DIR", ""), os.getenv("PATH", ""))

env: CLI=ngccli_cat_linux.zip
--2022-02-24 15:31:12--  https://ngc.nvidia.com/downloads/ngccli_cat_linux.zip
Resolving ngc.nvidia.com (ngc.nvidia.com)... 108.156.91.12, 108.156.91.42, 108.156.91.29, ...
Connecting to ngc.nvidia.com (ngc.nvidia.com)|108.156.91.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30994260 (30M) [application/zip]
Saving to: ‘local_project_dir/ngccli/ngccli_cat_linux.zip’

ngccli_cat_linux.zi 100%[===================>]  29.56M  47.0MB/s    in 0.6s    

2022-02-24 15:31:13 (47.0 MB/s) - ‘local_project_dir/ngccli/ngccli_cat_linux.zip’ saved [30994260/30994260]

Archive:  local_project_dir/ngccli/ngccli_cat_linux.zip
  inflating: local_project_dir/ngccli/ngc  
 extracting: local_project_dir/ngccli/ngc.md5  


In [26]:
!ngc registry model list nvidia/tao/pretrained_classification:*

+-------+-------+-------+-------+-------+-------+-------+-------+-------+
| Versi | Accur | Epoch | Batch | GPU   | Memor | File  | Statu | Creat |
| on    | acy   | s     | Size  | Model | y Foo | Size  | s     | ed    |
|       |       |       |       |       | tprin |       |       | Date  |
|       |       |       |       |       | t     |       |       |       |
+-------+-------+-------+-------+-------+-------+-------+-------+-------+
| vgg19 | 77.56 | 80    | 1     | V100  | 153.7 | 153.7 | UPLOA | Aug   |
|       |       |       |       |       |       | 2 MB  | D_COM | 18,   |
|       |       |       |       |       |       |       | PLETE | 2021  |
| vgg16 | 77.17 | 80    | 1     | V100  | 113.2 | 113.1 | UPLOA | Aug   |
|       |       |       |       |       |       | 6 MB  | D_COM | 18,   |
|       |       |       |       |       |       |       | PLETE | 2021  |
| squee | 65.13 | 80    | 1     | V100  | 6.5   | 6.46  | UPLOA | Aug   |
| zenet |       |       |       |     

In [27]:
!mkdir -p $LOCAL_EXPERIMENT_DIR/pretrained_resnet18/

In [28]:
# Pull pretrained model from NGC
!ngc registry model download-version nvidia/tao/pretrained_classification:resnet18 --dest $LOCAL_EXPERIMENT_DIR/pretrained_resnet18

Downloaded 82.38 MB in 6s, Download speed: 13.71 MB/s               
----------------------------------------------------
Transfer id: pretrained_classification_vresnet18 Download status: Completed.
Downloaded local path: /home/kmilo8346/pactolus_belt_vision/classification/local_project_dir/classification/pretrained_resnet18/pretrained_classification_vresnet18
Total files downloaded: 1 
Total downloaded size: 82.38 MB
Started at: 2022-02-24 15:33:18.477688
Completed at: 2022-02-24 15:33:24.486819
Duration taken: 6s
----------------------------------------------------


In [29]:
print("Check that model is downloaded into dir.")
!ls -l $LOCAL_EXPERIMENT_DIR/pretrained_resnet18/pretrained_classification_vresnet18

Check that model is downloaded into dir.
total 91096
-rw------- 1 kmilo8346 kmilo8346 93278448 Feb 24 15:33 resnet_18.hdf5


In [31]:
# checking specs for train
!cat $LOCAL_SPECS_DIR/stage1_classification_spec.cfg

model_config {
  arch: "resnet",
  n_layers: 18
  # Setting these parameters to true to match the template downloaded from NGC.
  use_batch_norm: true
  all_projections: true
  freeze_blocks: 0
  freeze_blocks: 1
  input_image_size: "2,073,600"
}
train_config {
  train_dataset_path: "/workspace/tao-experiments/data/stage1_training_split/train"
  val_dataset_path: "/workspace/tao-experiments/data/stage1_training_split/val"
  pretrained_model_path: "/workspace/tao-experiments/classification/pretrained_resnet18/pretrained_classification_vresnet18/resnet_18.hdf5"
  optimizer {
    sgd {
    lr: 0.01
    decay: 0.0
    momentum: 0.9
    nesterov: False
  }
}
  batch_size_per_gpu: 64
  n_epochs: 80
  n_workers: 16
  preprocess_mode: "caffe"
  enable_random_crop: True
  enable_center_crop: True
  label_smoothing: 0.0
  mixup_alpha: 0.1
  # regularizer
  reg_config {
    type: "L2"
    scope: "Conv2D,Dense"
    weight_decay: 0.00005
  }

  # learning_rate
  lr_config {
    step {
      learnin

#### stage1 training

In [ ]:
# stage 1 train!!
!tao classification train -e $SPECS_DIR/stage1_classification_spec.cfg -r $USER_EXPERIMENT_DIR/stage1_output -k $KEY

#### stage1 evaluate

In [4]:
!tao classification evaluate -e $SPECS_DIR/stage1_classification_spec.cfg -k $KEY

2022-02-24 19:57:52,589 [INFO] root: Registry: ['nvcr.io']
2022-02-24 19:57:52,646 [INFO] tlt.components.instance_handler.local_instance: Running command in container: nvcr.io/nvidia/tao/tao-toolkit-tf:v3.21.11-tf1.15.5-py3
Matplotlib created a temporary config/cache directory at /tmp/matplotlib-r9p992pr because the default path (/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Using TensorFlow backend.
Using TensorFlow backend.

2022-02-24 19:58:02,149 [WARNING] tensorflow: From /root/.cache/bazel/_bazel_root/ed34e6d125608f91724fda23656f1726/execroot/ai_infra/bazel-out/k8-fastbuild/bin/magnet/packages/iva/build_wheel.runfiles/ai_infra/iva/makenet/scripts/evaluate.py:28: The name tf.ConfigProto is deprecated. Please use tf.compat.v1.ConfigProto instead.


2022-02-24 19:58:02,149 [WARNING] tensorflow: Fr

### 4. PRUNE && RE-TRAIN

#### prunning best (027)

In [5]:
# Defined epoch 27 as best after evaluations exploration
%env EPOCH=027
!mkdir -p $LOCAL_EXPERIMENT_DIR/stage1_output/resnet_pruned
!tao classification prune -m $USER_EXPERIMENT_DIR/stage1_output/weights/resnet_$EPOCH.tlt \
           -o $USER_EXPERIMENT_DIR/stage1_output/resnet_pruned/resnet18_nopool_bn_pruned.tlt \
           -eq union \
           -pth 0.6 \
           -k $KEY \
           --results_dir $USER_EXPERIMENT_DIR/logs

env: EPOCH=027
2022-02-24 20:16:42,354 [INFO] root: Registry: ['nvcr.io']
2022-02-24 20:16:42,428 [INFO] tlt.components.instance_handler.local_instance: Running command in container: nvcr.io/nvidia/tao/tao-toolkit-tf:v3.21.11-tf1.15.5-py3
Matplotlib created a temporary config/cache directory at /tmp/matplotlib-5v9xwifm because the default path (/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Using TensorFlow backend.
Using TensorFlow backend.
2022-02-24 20:16:56,314 [INFO] modulus.pruning.pruning: Exploring graph for retainable indices
2022-02-24 20:16:56,967 [INFO] modulus.pruning.pruning: Pruning model and appending pruned nodes to new graph
2022-02-24 20:17:15,157 [INFO] iva.common.magnet_prune: Pruning ratio (pruned model / original model): 0.868215683213357
2022-02-24 20:17:17,401 [INFO] tlt.compo

In [8]:
print('Pruned model:')
print('------------')
!ls -rlt $LOCAL_EXPERIMENT_DIR/stage1_output/resnet_pruned

Pruned model:
------------
total 39460
-rw-r--r-- 1 kmilo8346 kmilo8346 40406480 Feb 24 20:17 resnet18_nopool_bn_pruned.tlt


#### retrain pruned model

In [10]:
!cat $LOCAL_SPECS_DIR/stage1_classification_retrain_spec.cfg

model_config {
  arch: "resnet",
  n_layers: 18
  use_batch_norm: true
  all_projections: true
  input_image_size: "3,224,224"
}
train_config {
  train_dataset_path: "/workspace/tao-experiments/data/stage1_trainig_split/train"
  val_dataset_path: "/workspace/tao-experiments/data/stage1_trainig_split/val"
  pretrained_model_path: "/workspace/tao-experiments/classification/stage1_output/resnet_pruned/resnet18_nopool_bn_pruned.tlt"
  optimizer {
    sgd {
    lr: 0.01
    decay: 0.0
    momentum: 0.9
    nesterov: False
  }
}
  batch_size_per_gpu: 64
  n_epochs: 80
  n_workers: 16
  preprocess_mode: "caffe"
  enable_random_crop: True
  enable_center_crop: True
  label_smoothing: 0.0
  mixup_alpha: 0.1
  # regularizer
  reg_config {
    type: "L2"
    scope: "Conv2D,Dense"
    weight_decay: 0.00005
  }

  # learning_rate
  lr_config {
    step {
      learning_rate: 0.006
      step_size: 10
      gamma: 0.1
    }
  }
}
eval_config {
  eval_dataset_path: "/workspace/tao-experiments/data/st

In [ ]:
!tao classification train -e $SPECS_DIR/stage1_classification_retrain_spec.cfg \
                      -r $USER_EXPERIMENT_DIR/stage1_output_retrain \
                      -k $KEY

### 5. EVALUATE BEST

In [5]:
# got 046 as best acc
!tao classification evaluate -e $SPECS_DIR/stage1_classification_retrain_spec.cfg -k $KEY

2022-02-24 23:41:46,693 [INFO] root: Registry: ['nvcr.io']
2022-02-24 23:41:46,754 [INFO] tlt.components.instance_handler.local_instance: Running command in container: nvcr.io/nvidia/tao/tao-toolkit-tf:v3.21.11-tf1.15.5-py3
Matplotlib created a temporary config/cache directory at /tmp/matplotlib-1v833_0s because the default path (/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Using TensorFlow backend.
Using TensorFlow backend.

2022-02-24 23:41:56,045 [WARNING] tensorflow: From /root/.cache/bazel/_bazel_root/ed34e6d125608f91724fda23656f1726/execroot/ai_infra/bazel-out/k8-fastbuild/bin/magnet/packages/iva/build_wheel.runfiles/ai_infra/iva/makenet/scripts/evaluate.py:28: The name tf.ConfigProto is deprecated. Please use tf.compat.v1.ConfigProto instead.


2022-02-24 23:41:56,045 [WARNING] tensorflow: Fr